<a href="https://jupyterhub.user.eopf.eodc.eu/hub/login?next=%2Fhub%2Fspawn%3Fnext%3D%252Fhub%252Fuser-redirect%252Fgit-pull%253Frepo%253Dhttps%253A%252F%252Fgithub.com%252Feopf-toolkit%252Feopf-101%2526branch%253Dmain%2526urlpath%253Dlab%252Ftree%252Feopf-101%252F05_zarr_tools%252F57_zarr_R_gdal.ipynb%23fancy-forms-config=%7B%22profile%22%3A%22choose-your-environment%22%2C%22image%22%3A%22unlisted_choice%22%2C%22image%3Aunlisted_choice%22%3A%224zm3809f.c1.de1.container-registry.ovh.net%2Feopf-toolkit-r%2Feopf-toolkit-r%3Alatest%22%2C%22autoStart%22%3A%22true%22%7D" target="_blank">
<button style="background-color:#0072ce; color:white; padding:0.6em 1.2em; font-size:1rem; border:none; border-radius:6px; margin-top:1em;">
🚀 Launch this notebook in JupyterLab
</button>
</a>

### Introduction

This notebook demonstrates how to **retrieve** remotely stored Zarr data using the Zarr GDAL driver in R. We will explore how to **read and visualize** Zarr data (zarrays) and their metadata using the `stars`, `sf` and `terra` packages as well as their current limitations. 

::: {.callout-tip}
This notebook has a sibling, which demonstrates how to access the same Zarr data using the `Rarr` package. You can find it [here](./54_zarr_R_rarr.qmd).
:::

### What we will learn

- ✏️ How to edit URLs of Zarr stores to make them readable for GDAL
- 🔎 Which read-functions and arguments to use in `stars` and `terra`
- 🚧 Current limitations of these packages

### Prerequisites

To follow this notebook, you will need to load a STAC item in Zarr format from the [EOPF Zarr STAC Catalog](https://stac.browser.user.eopf.eodc.eu/?.language=en). If you are new to STAC inside the R environment, we suggest to follow the [Access the EOPF Zarr STAC API with R tutorial](https://eopf-toolkit.github.io/eopf-101/51_eopf_stac_r.html). The item we are using for this example mirrors the one used in the [Sentinel-2 L1C MSI Zarr Product Exploration notebook](https://eopf-sample-service.github.io/eopf-sample-notebooks/sentinel-2-l1c-msi-zarr-product-exploration/#introduction) which is hosted on EODC's object storage and is accessible under the URL below.

In [ ]:
zarr_url = "https://objects.eodc.eu/e05ab01a9d56408d82ac32d69a5aae2a:sample-data/tutorial_data/cpm_v253/S2B_MSIL1C_20250113T103309_N0511_R108_T32TLQ_20250113T122458.zarr"

### Import packages

In [ ]:
library(stars)     # reading; imports sf too
library(terra)     # reading
library(jsonlite)  # parsing metadata
library(mapview)   # interactive maps

## GDAL's Zarr driver

Common R-packages for raster data like `stars` and `terra` can leverage GDAL's raster drivers to read data directly from remote locations. This is done using GDAL's virtual file system (VSI) capabilities, specifically the `/vsicurl/` prefix for accessing files over HTTP(S). 

For Zarr data GDAL provides a dedicated Zarr driver that requires an additional `ZARR:`-prefix to the VSI URL. Finally, the complete prefix needs to be quoted: `"ZARR:/vsicurl/"`.

In [ ]:
vsi_prefix = "ZARR:/vsicurl/"
vsi_url = paste0(vsi_prefix, dplyr::as_label(zarr_url)) # needs special quoting
print(vsi_url)

## Metadata exploration with GDAL

With the VSI URL pointing to the remote Zarr store we can now use GDAL utilities to explore the metadata of the Zarr store. The `sf::gdal_utils` function provides an interface to GDAL command-line utilities. We use the `mdiminfo` command to retrieve metadata about the multidimensional arrays (zarrays) contained within the Zarr store. We only print the top-level names of the arrays here.

In [ ]:
metadata = sf::gdal_utils("mdiminfo", source = vsi_url, quiet = T) |> 
  fromJSON()
names(metadata)

::: {.callout-tip collapse="true"}
## Expand for full GDAL MDIM metadata output

In [ ]:
print(metadata)

:::


## Reading Zarr data with `stars` and `terra`

::: {.callout-important}
In this section we show the current capabilities and limitations of `stars` and `terra` when it comes to reading Zarr array(s). We test the functions `read_stars` and `read_mdim` from `stars` and `rast` from the `terra`package. 
Successful attempts are marked with a ✅, failed attempts with a ⛔.
:::

First we define the path to a specific data array (band 1, 60 meter resolution) inside the Zarr store which can be understood as a single raster layer.

In [ ]:
band_variable = "/measurements/reflectance/r60m/b01"

### `stars::read_stars()`

⛔ Traditional approach for reading all bands/layers and specifying a driver.

In [ ]:
r = read_stars(vsi_url, driver = "ZARR")

⛔ Traditional approach for reading all bands/layers and specifying a sub-dataset (integer).

In [ ]:
r = read_stars(vsi_url, sub = 1)

⛔ Traditional approach for reading all bands/layers and specifying a sub-dataset (path).

In [ ]:
r = read_stars(vsi_url, sub = band_variable)

✅ Constructing the full zarray path from prefix, URL, and band variable.

In [ ]:
(r = read_stars(paste(vsi_url, band_variable, sep = ":")))
st_crs(r) # NA: empty

This method successfully reads the specified band from the remote Zarr store. 

:::{.callout-warning}
Note that the coordinate reference system (CRS) is not automatically recognized and needs to be set manually. We can look it up in the metadata where it is listed as one of the STAC properties.
:::

In [ ]:
(crs = metadata$attributes$stac_discovery$properties$`proj:epsg`)
st_crs(r) = crs 

Our data is in CRS _WGS 84 / UTM zone 32N (EPSG:32632)_.
We can now safely plot it as a static map...

In [ ]:
system.time(plot(r, axes = TRUE)) 

... or is an interactive visualization:

In [ ]:
mapviewOptions(basemaps = c("OpenTopoMap","Esri.WorldImagery", 
                            "CartoDB.Positron"))

In [ ]:
mapview(r)

---

### `stars::read_mdim()`

⛔ Traditional approach for reading all bands/layers at once. "?" should return a list of array possible names.

In [ ]:
m = read_mdim(vsi_url)
m = read_mdim(vsi_url, variable = "?") # query array names

✅ Specifying the full zarray path from prefix, URL, and band variable. Read as a proxy object (just metadata; values are accessed lazily when required).

In [ ]:
system.time({
  (m = read_mdim(vsi_url, variable = band_variable, proxy = TRUE))
  }) # fast: only reads metadata

st_crs(m) # NA: empty
system.time(plot(m, axes = T)) # slow: only here the full array is downloaded

As for `read_stars()`, the CRS is not automatically recognized and needs to be set manually.

---

### `terra::rast()`

⛔ Traditional approach for reading all bands/layers at once.

In [ ]:
(tr = terra::rast(vsi_url))
tr$b02

names(tr) # has non-unique names, e.g. three times "b02"

`terra` reads some of the available bands, but fails to name them uniquely. It likely searches for band names like "b02" and finds the first group of data arrays by order, in this case the 10-meter bands (2, 3, 4, and 8 aka R,G,B, and NIR). The Zarr store contains multiple arrays named after each band such as `/measurements/reflectance/r10m/b02`, a corresponding mask (e.g. `quality/mask/r10m/b02`) and the detector footprint (e.g. `/conditions/mask/detector_footprint/r10m/b02`). All of these zarrays share the same name, `b02`, leading to non-unique layer names in the resulting `SpatRaster` object. When constructing unique names we can at least visualize the data.

In [ ]:
# rename layers to (not meaningful) unique names
names(tr) = paste0(letters[1:nlyr(tr)], names(tr)) 
names(tr)

plot(tr)

✅ Specifying the full zarray path from prefix, URL, and band variable as sub-dataset. Read as proxy object.

In [ ]:
system.time({
  tr = rast(vsi_url, subds = band_variable)
  })# fast: only reads metadata

crs(tr) # "": empty

As for the `stars`-functions, the CRS is not automatically recognized and needs to be set manually.

In [ ]:
system.time(plot(tr)) # slow: only here the full array is downloaded 

## 💪 Now it is your turn

- 🔍 **Task**: Explore the metadata of other Zarr stores using GDAL's `mdiminfo` command via `sf::gdal_utils()`. Store the JSON output in an R-object and find entries that store relevant metadata, such as the CRS, bounding box, or sensor-related information.
- 📖 Read the [GDAL Zarr driver documentation](https://gdal.org/en/stable/drivers/raster/zarr.html) to learn more about opening options.

## Conclusion

In this notebook we have demonstrated how to access remote Zarr stores using GDAL's Zarr driver in R with the `stars` and `terra` packages. We have seen that while it is possible to read specific data arrays, there are some limitations, such as the lack of automatic CRS recognition and challenges with reading multiple arrays simultaneously.

## What's next?

Explore [this related notebook](./56_zarr_R_rarr.qmd) which demonstrates how to access the same Zarr data using the `Rarr` package and building a `stars` object from it.